In [1]:
import numpy as np
from ase import Atoms
from collections import Counter

TYPE_TO_ELEMENT = {13: "Al", 27: "Co", 28: "Ni"}

def parse(path):
    lines = open(path).read().splitlines()
    cell_rows, body_start = [], None
    for i, ln in enumerate(lines):
        if ln.strip() == "":
            continue
        cell_rows.append([float(x) for x in ln.split()[:3]])
        if len(cell_rows) == 3:
            body_start = i + 1
            break
    cell = np.array(cell_rows)
    symbols, scaled, dropped = [], [], 0
    for ln in lines[body_start:]:
        s = ln.strip()
        if s == "" or s.startswith("#"):
            continue
        parts = s.split()
        if len(parts) >= 2 and parts[1] == "#":
            continue
        if len(parts) < 5:
            continue
        try:
            x, y, z = float(parts[0]), float(parts[1]), float(parts[2])
            typ = int(parts[3])
        except ValueError:
            continue
        if typ == 0:
            dropped += 1
            continue
        if typ in TYPE_TO_ELEMENT:
            symbols.append(TYPE_TO_ELEMENT[typ]); scaled.append([x, y, z])
    return Atoms(symbols=symbols, scaled_positions=np.array(scaled), cell=cell, pbc=True), dropped

def parse_fig5(path):
    lines = open(path).read().splitlines()
    text = open(path).read().split()
    cell = np.array([float(x) for x in text[:9]]).reshape(3, 3)
    symbols, scaled = [], []
    for ln in lines:
        parts = ln.split()
        if len(parts) >= 6 and parts[3].isdigit() and parts[4].isdigit():
            x, y, z = float(parts[0]), float(parts[1]), float(parts[2])
            typ = int(parts[3])
            if typ == 0:
                continue
            if typ in TYPE_TO_ELEMENT:
                symbols.append(TYPE_TO_ELEMENT[typ]); scaled.append([x, y, z])
    return Atoms(symbols=symbols, scaled_positions=np.array(scaled), cell=cell, pbc=True)

print("Parsers ready.")

Parsers ready.


In [2]:
from ase.optimize import BFGS
from ase.filters import FrechetCellFilter

def relax_with_diagnostics(structure_name, n_atoms, model_name, calc, base_atoms):
    a = base_atoms.copy()
    a.calc = calc
    e_initial = a.get_potential_energy()
    f_initial = a.get_forces()
    max_force_initial = np.sqrt((f_initial**2).sum(axis=1)).max()
    v0 = a.get_volume()
    p0 = a.get_positions().copy()
    opt = BFGS(FrechetCellFilter(a), logfile=None)
    opt.run(fmax=0.01)
    e_final = a.get_potential_energy()
    n = len(a)
    disp = np.sqrt(((a.get_positions() - p0)**2).sum(axis=1))
    return {
        "structure": structure_name, "n_atoms": n_atoms, "model": model_name,
        "dE_per_atom_meV": round(1000*(e_final - e_initial)/n, 1),
        "max_force_initial": round(max_force_initial, 3),
        "vol_change_pct": round(100*(a.get_volume()-v0)/v0, 2),
        "max_disp_A": round(disp.max(), 3),
    }

print("Diagnostics function ready.")

Diagnostics function ready.


In [4]:
from sevenn.calculator import SevenNetCalculator
calc_7net = SevenNetCalculator(model="7net-0", device="cpu")

def get_atoms(parse_result):
    if isinstance(parse_result, tuple):
        return parse_result[0]
    return parse_result

sevennet_results = []

# 26-atom
a26 = get_atoms(parse_fig5("Al9Co2Ni2-coords.txt"))
sevennet_results.append(relax_with_diagnostics("Al9Co2Ni2", 26, "SevenNet-0", calc_7net, a26))
print("26-atom done")

# 60-atom
a60 = get_atoms(parse("Al2CoNi-coords.txt"))
sevennet_results.append(relax_with_diagnostics("Al2CoNi", 60, "SevenNet-0", calc_7net, a60))
print("60-atom done")

# 265-atom W-phase
a265 = get_atoms(parse("W-AlCoNi-coords"))
sevennet_results.append(relax_with_diagnostics("W-AlCoNi", 265, "SevenNet-0", calc_7net, a265))
print("265-atom done")

import json
with open("o1_sevennet_results.json", "w") as f:
    json.dump(sevennet_results, f, indent=2)

import pandas as pd
pd.DataFrame(sevennet_results)

/opt/anaconda3/envs/sevennet-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/anaconda3/envs/sevennet-env/lib/python3.11/site-packages/sevenn/calculator.py:93: UserWarning: No tensor product accelerator is enabled for SevenNetCalculator. SevenNet may run much slower without a TP accelerator. Please refer to the accelerator section of the documentation.
  util.warn_no_tp_accelerator('SevenNetCalculator')


26-atom done


/opt/anaconda3/envs/sevennet-env/lib/python3.11/site-packages/scipy/_lib/_util.py:1181: RuntimeWarning: logm result may be inaccurate, approximate err = 6.226722841044345e-13
  return f(*arrays, *other_args, **kwargs)
/opt/anaconda3/envs/sevennet-env/lib/python3.11/site-packages/scipy/_lib/_util.py:1181: RuntimeWarning: logm result may be inaccurate, approximate err = 6.257610945066841e-13
  return f(*arrays, *other_args, **kwargs)
/opt/anaconda3/envs/sevennet-env/lib/python3.11/site-packages/scipy/_lib/_util.py:1181: RuntimeWarning: logm result may be inaccurate, approximate err = 6.292370866369317e-13
  return f(*arrays, *other_args, **kwargs)
/opt/anaconda3/envs/sevennet-env/lib/python3.11/site-packages/scipy/_lib/_util.py:1181: RuntimeWarning: logm result may be inaccurate, approximate err = 6.319963733289555e-13
  return f(*arrays, *other_args, **kwargs)
/opt/anaconda3/envs/sevennet-env/lib/python3.11/site-packages/scipy/_lib/_util.py:1181: RuntimeWarning: logm result may be inacc

60-atom done
265-atom done


TypeError: Object of type float32 is not JSON serializable

In [5]:
import json

# convert any numpy float32/float64 to plain Python floats so JSON can save them
clean = [{k: (float(v) if hasattr(v, "item") else v) for k, v in row.items()}
         for row in sevennet_results]

with open("o1_sevennet_results.json", "w") as f:
    json.dump(clean, f, indent=2)
print("Saved", len(clean), "rows")

import pandas as pd
pd.DataFrame(clean)

Saved 3 rows


,structure,n_atoms,model,dE_per_atom_meV,max_force_initial,vol_change_pct,max_disp_A
0,Al9Co2Ni2,26,SevenNet-0,-2.0,0.198,0.21,0.161
1,Al2CoNi,60,SevenNet-0,-1.9,0.335,0.38,0.060
2,W-AlCoNi,265,SevenNet-0,-2.4,0.316,0.40,0.583


In [6]:
import json
import pandas as pd

# load all three models' results from their saved JSON files
sevennet = json.load(open("o1_sevennet_results.json"))

# MACE results (from o1_table_v2.json if you saved it, else rebuild)
try:
    mace = json.load(open("o1_table_v2.json"))
except FileNotFoundError:
    mace = []  # tell me if this happens

# MatterSim was only run on the W-phase (265) earlier
# Its result is in your original o1_results.json
orig = json.load(open("o1_results.json"))
mattersim = [r for r in orig if "MatterSim" in r.get("model", "")]

# combine
all_rows = mace + sevennet
# (MatterSim row has different columns from the old format - we'll note it separately)

df = pd.DataFrame(all_rows)

# add kappa_SRME context (from Matbench Discovery leaderboard)
ksrme = {"MACE-MPA-0": 0.412, "SevenNet-0": 0.55, "MatterSim-v1-1M": 0.575}
df["kappa_SRME"] = df["model"].map(ksrme)

# sort by structure size then model
df = df.sort_values(["n_atoms", "model"])
df

,structure,n_atoms,model,dE_per_atom_meV,max_force_initial,vol_change_pct,max_disp_A,kappa_SRME
0,Al9Co2Ni2,26,MACE-MPA-0,-1.7,0.177,-1.13,0.060,0.412
3,Al9Co2Ni2,26,SevenNet-0,-2.0,0.198,0.21,0.161,0.550
1,Al2CoNi,60,MACE-MPA-0,-1.3,0.330,0.31,0.045,0.412
4,Al2CoNi,60,SevenNet-0,-1.9,0.335,0.38,0.060,0.550
2,W-AlCoNi,265,MACE-MPA-0,-1.9,0.297,-1.03,0.471,0.412
5,W-AlCoNi,265,SevenNet-0,-2.4,0.316,0.40,0.583,0.550
